# Machine Learning Data Preparation
In this notebook we scale features and extract principal components of users for unsupervised clustering in the users ML segmentation notebook

## Bootstrap
Configuring parameters, loading libraries and dataset.

In [331]:
import sys, os
# This line allows us to import from the parent directory, which is where the 'src' folder is located.
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
# These lines enable automatic reloading of modules in Jupyter, so that changes to the code are reflected without needing to restart the kernel.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Libraries
External

In [332]:
import numpy as np
import pandas as pd
import warnings
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler

Project libraries

In [333]:
from src.utils import Config

### Configuration
Set here general paramenters

In [334]:
config = Config({
    'datasets_path': '../data/aggregated/',
    'save_csvs_path': '../data/processed/',
    'pca_retained_variance': 0.95,
})

### Loading Datasets

In [335]:
users = pd.read_csv(f"{config.datasets_path}/users.csv")
# Convert Users date columns to datetime format
cols_to_datetime = ['birthdate', 'sign_up_date']
users[cols_to_datetime] = users[cols_to_datetime].apply(pd.to_datetime, errors='coerce')

At this point we will remove all columns that do not represent relevant features for us:
`user_id`, `birthdate`, `home_city`, `home_airport`, `sign_up_date`, `total_sesions`

In [336]:
columns_to_remove = ['user_id', 'birthdate', 'home_city', 'home_airport', 'sign_up_date', 'total_sesions']
print("Removing features:", columns_to_remove)
users_scaled = users.drop(columns=columns_to_remove, axis=1)

Removing features: ['user_id', 'birthdate', 'home_city', 'home_airport', 'sign_up_date', 'total_sesions']


## Imputation for missing values

In [337]:
features_count_null = users_scaled.isnull().sum()
missing_values = features_count_null[features_count_null > 0]
print("Missing values in users dataset:")
for (feature, count) in missing_values.items():
  print(f"  {feature:<30} {count}")

Missing values in users dataset:
  avg_page_clicks_flight_booked  826
  avg_page_clicks_hotel_booked   597
  avg_seats_booked               1042
  avg_flight_amount              1042
  avg_flight_discount            4362
  avg_cents_km_flown             1042
  avg_hotel_amount               662
  avg_hotel_discount             4382
  avg_hotel_nights               662
  avg_checked_bags               1042
  avg_distance_flown             1042
  avg_trip_lead_time             459
  avg_trip_duration              1096
  pct_only_flight_booked         459
  pct_only_hotel_booked          459
  pct_return_flight_booked       826
  weekends_ratio                 1096


Those missing values are all for users that do not have a flight or hotel booked. Therefore, we'll replace `NaN` by `0`, which is meaningful here.

In [338]:
for feature in missing_values.index:
    users_scaled.loc[users[feature].isnull(), feature] = 0

## Feature Encoding

## Feature Scaling
In this section we analize the users' features and choose the best scaling method for each of them.

### Categorical Features
For categorical data, we'll map the values as the following:
* `gender`: as "Other" is a very seldom value (0.2%), we'll simply set `0` for values of `"F"` (88.2%) and `1`for other values, `["M","O"]`, (11.8%)
* `married`: we'll set `0` for `False` (56%) and `1` for `True` (44%)
* `has_children`: similarly, we'll set set `0` for `False` (67.4%) and `1` for `True` (32.6%)
* `home_counry`: has only two values, which we will attribute `0` for `"usa"` and `1` for `"canada"`

In [339]:
users_scaled['gender'] = (users_scaled['gender'] != 'F').astype(int)  # 0 for 'F', 1 for others
users_scaled['married'] = users_scaled['married'].astype(int)  # 0 for False, 1 for True
users_scaled['has_children'] = users_scaled['has_children'].astype(int)  # 0 for False, 1 for True
users_scaled['home_country'] = (users_scaled['home_country'] != 'usa').astype(int)  # 0 for 'usa', 1 for others_users = users.copy()

### Numerical Features


In [340]:
# We scale using a StandarScaler then a MinMaxScaler to ensure all features are on the same scale, which is important for PCA and clustering algorithms.
for feature in users_scaled.columns:
    users_scaled[feature] = MinMaxScaler().fit_transform(StandardScaler().fit_transform(users_scaled[[feature]])).flatten()

Final dataset with all features scaled:

In [341]:
users_scaled.head()

,gender,married,has_children,home_country,home_airport_lat,home_airport_lon,age,account_age,total_trips,flight_bookings,...,avg_distance_flown,total_distance_flown,avg_trip_lead_time,avg_trip_duration,weekdays_travelled,weekends_travelled,pct_only_flight_booked,pct_only_hotel_booked,pct_return_flight_booked,weekends_ratio
0,0.0,1.0,0.0,0.0,0.487317,0.890149,0.676056,1.000000,0.250,0.000,...,0.000000,0.000000,0.032967,0.000000,0.000000,0.000000,0.0,1.0,0.0,0.000000
1,0.0,1.0,0.0,0.0,0.450257,0.669431,0.492958,0.699248,0.250,0.250,...,0.077628,0.062285,0.017857,0.057692,0.040816,0.055556,0.0,0.0,1.0,0.333333
2,0.0,1.0,1.0,0.0,0.646601,0.375429,0.478873,0.684211,0.250,0.125,...,0.051573,0.020690,0.020604,0.153846,0.040816,0.111111,0.0,0.5,1.0,0.500000
3,0.0,1.0,0.0,0.0,0.527056,0.920511,0.366197,0.684211,0.625,0.625,...,0.070639,0.141693,0.015385,0.184615,0.285714,0.555556,0.0,0.0,1.0,0.416667
4,0.0,1.0,1.0,0.0,0.113835,0.815754,0.394366,0.673684,0.125,0.125,...,0.714887,0.573586,0.545330,0.500000,0.367347,0.444444,0.0,0.0,1.0,0.307692


## Principal Components Analysis
Here we generate a dataset based on the principal components reaching the information level defined in `config.pca_retained_variance`.

In [342]:
# Perform PCA to retain config.pca_retained_variance of the variance
pca = PCA(n_components=config.pca_retained_variance)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    users_pca = pca.fit_transform(users_scaled)

# Create a DataFrame with the principal components
users_pca = pd.DataFrame(users_pca, columns=[f'PC{i+1}' for i in range(users_pca.shape[1])])

# Display the explained variance ratio
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {pca.explained_variance_ratio_.cumsum()}")
print(f"Number of components: {pca.n_components_}")

Explained variance ratio: [0.20579917 0.18443765 0.13011727 0.10181739 0.06905385 0.06522568
 0.0476562  0.03639095 0.02425146 0.02322385 0.01644298 0.01516326
 0.01414008 0.01010066 0.00935757]
Cumulative explained variance: [0.20579917 0.39023682 0.52035409 0.62217149 0.69122533 0.75645101
 0.80410721 0.84049816 0.86474962 0.88797348 0.90441645 0.91957971
 0.93371979 0.94382045 0.95317801]
Number of components: 15


In [343]:
users_pca.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15
0,-0.973710,0.360205,-0.763463,-0.212338,-0.213849,0.641384,0.373454,0.157312,0.250989,-0.011743,-0.041586,0.083411,0.221480,0.075194,0.086436
1,0.165021,0.318486,-0.555241,-0.227127,-0.032670,-0.245539,-0.115623,-0.286845,-0.014782,-0.040630,-0.034209,0.062226,0.001349,-0.135746,0.049390
2,-0.102104,0.881264,0.240389,-0.168082,-0.154946,0.004552,0.363962,-0.266275,-0.247478,0.052130,0.136726,0.176884,0.023107,-0.134813,0.272391
3,0.859261,0.227643,-0.518099,0.092133,-0.138295,0.337910,-0.238784,-0.001616,0.260364,-0.079827,0.245979,-0.057991,-0.080255,-0.026982,0.115768
4,0.574514,0.913029,0.414717,-0.275671,0.012153,-0.643822,0.317798,0.724480,-0.140662,-0.822549,0.097476,-0.496096,0.239770,-0.371047,-0.332791


## Saving Datasets

In [344]:
# Keeping reference to user_id
users_scaled['user_id'] = users['user_id']
users_pca['user_id'] = users['user_id']

# Saving CSVs
users_scaled.to_csv(config.save_csvs_path + f'users_scaled.csv', index=False)
users_pca.to_csv(config.save_csvs_path + f'users_pca.csv', index=False)
print("CSVs saved successfully in " + config.save_csvs_path)

CSVs saved successfully in ../data/processed/
